# See Water through the Years and Seasons

Follow annual and seasonal rainfall, evapotranspiration and runoff, then read fortnightly water and vegetation values.

Run the cells in order. Each step uses data from the previous cells. You can edit the place, identifier, columns and chart settings as you go.


## Set up Python

Run these two collapsed cells once. They load the libraries and starting location. Expand them to see or change the setup.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["geopandas", "matplotlib", "requests", "pyodide-http"])
    import pyodide_http
    pyodide_http.patch_all()

import os
import re
import ast
import json
from getpass import getpass
from urllib.parse import urljoin
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink


In [ ]:
SCOPE = json.loads("{\"state\": \"Bihar\", \"district\": \"Nalanda\", \"tehsil\": \"Hilsa\"}")
GEOSERVER = 'https://geoserver.core-stack.org:8443/geoserver/'
API_URL = 'https://geoserver.core-stack.org/api/v1/'
STAC_URL = 'https://spatio-temporal-asset-catalog.s3.ap-south-1.amazonaws.com/CorestackCatalogs_merged_collection/tehsil_wise/catalog.json'
YEARS = list(range(2017, 2025))


## Choose the tehsil

The template defaults to Hilsa, Nalanda, Bihar. GeoLibre downloads use your selected place instead. Edit `SCOPE` in the setup cell to change tehsil, then restart the kernel and run from the top. Layer coverage can differ between places.


In [ ]:
state = re.sub(r"[\s_]+", "_", SCOPE["state"].replace("(", "").replace(")", "")).strip("_").lower()
district = re.sub(r"[\s_]+", "_", SCOPE["district"].replace("(", "").replace(")", "")).strip("_").lower()
tehsil = re.sub(r"[\s_]+", "_", SCOPE["tehsil"].replace("(", "").replace(")", "")).strip("_").lower()
place = {"state": state, "district": district, "tehsil": tehsil}
place


## Set your API key for this session

The [public API guide](https://docs.core-stack.org/use-precomputed-data/public-apis/) explains how to register, generate an API key and use it. The key goes in the `X-API-Key` header. This cell stores `CORE_STACK_API_KEY` in the current Python kernel’s environment, so later API cells can reuse it. An existing key is reused without prompting. You can skip this cell when exploring only GeoServer or STAC data. Restarting the kernel may require entering the key again.


In [ ]:
from inspect import isawaitable

api_key = os.environ.get("CORE_STACK_API_KEY", "").strip()
if not api_key:
    api_key = getpass("CoRE Stack API key: ")
    if isawaitable(api_key):
        api_key = await api_key

os.environ["CORE_STACK_API_KEY"] = str(api_key).strip()


## The layers we will use

Each link reads a vector layer as GeoJSON. GeoJSON contains a feature list; each feature has a shape and its data fields. The seasonal table is calculated from the fortnightly water layer. NDVI has separate crop, tree and shrub vector layers.


In [ ]:
layer_urls = {
    "annual_water": f"{GEOSERVER}mws_layers/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=mws_layers:deltaG_well_depth_{district}_{tehsil}&outputFormat=application/json&srsName=EPSG:4326",
    "fortnightly_water": f"{GEOSERVER}mws_layers/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=mws_layers:deltaG_fortnight_{district}_{tehsil}&outputFormat=application/json&srsName=EPSG:4326",
    "soge": f"{GEOSERVER}soge/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=soge:soge_vector_{district}_{tehsil}&outputFormat=application/json&srsName=EPSG:4326",
    "aquifer": f"{GEOSERVER}aquifer/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=aquifer:aquifer_vector_{district}_{tehsil}&outputFormat=application/json&srsName=EPSG:4326",
    "ndvi_crop": f"{GEOSERVER}ndvi_timeseries/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=ndvi_timeseries:ndvi_timeseries_{district}_{tehsil}_crop&outputFormat=application/json&srsName=EPSG:4326",
    "ndvi_tree": f"{GEOSERVER}ndvi_timeseries/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=ndvi_timeseries:ndvi_timeseries_{district}_{tehsil}_tree&outputFormat=application/json&srsName=EPSG:4326",
    "ndvi_shrub": f"{GEOSERVER}ndvi_timeseries/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=ndvi_timeseries:ndvi_timeseries_{district}_{tehsil}_shrub&outputFormat=application/json&srsName=EPSG:4326",
}
pd.DataFrame(layer_urls.items(), columns=["Layer", "GeoJSON URL"])


## Read annual water data

Each year is a JSON object stored in a field such as `2017_2018`. The object contains `Precipitation`, `ET` and `RunOff`, all in millimetres.


In [ ]:
annual_water_response = requests.get(layer_urls["annual_water"], timeout=90)
annual_water_response.raise_for_status()
annual_water_geojson = annual_water_response.json()
annual_water = gpd.GeoDataFrame.from_features(annual_water_geojson["features"], crs="EPSG:4326")
annual_water.drop(columns="geometry").head()


## Read the field descriptions

STAC records describe the published fields. This table selects the fields used below and keeps their original descriptions.


In [ ]:
item_name = f"{state}_{district}_{tehsil}_change_in_well_depth_vector"
item_url = urljoin(STAC_URL, f"{state}/{district}/{tehsil}/{item_name}/{item_name}.json")
item_response = requests.get(item_url, timeout=90)
item_response.raise_for_status()
item = item_response.json()
field_notes = pd.DataFrame(item["properties"]["table:columns"])
field_notes.loc[field_notes["name"].isin(['uid', '2017_2018', '2024_2025']), ["name", "type", "description"]]


## Choose one micro-watershed

`uid` is the MWS identifier in this layer. Start with the first one, or replace `mws_id` with another identifier from the displayed list.


In [ ]:
mws_ids = annual_water["uid"].sort_values().tolist()
display(pd.DataFrame({"MWS identifier": mws_ids}))
mws_id = mws_ids[0]
mws_id


## Make an annual table

Read the selected MWS, open each year’s JSON object and keep the three water values.


In [ ]:
water_row = annual_water.loc[annual_water["uid"] == mws_id].iloc[0]
annual_records = [json.loads(water_row[f"{year}_{year + 1}"]) for year in YEARS]
annual = pd.DataFrame(annual_records, index=YEARS)[["Precipitation", "ET", "RunOff"]]
annual = annual.rename(columns={"Precipitation": "Rainfall", "RunOff": "Runoff"})
annual.index.name = "July–June year starting"
annual


## Plot annual rainfall, ET and runoff

The three series share a millimetre axis. Change the figure size, colours or selected columns to try another presentation.


In [ ]:
ax = annual.plot(marker="o", figsize=(10, 4))
ax.set(title="Annual water balance", xlabel="July–June year starting", ylabel="Water depth (mm)")
ax.grid(axis="y", alpha=0.2)
plt.show()


## Read fortnightly water data

The date fields contain JSON objects. Use the published dates: interval starts can differ between tehsils.


In [ ]:
fortnightly_water_response = requests.get(layer_urls["fortnightly_water"], timeout=90)
fortnightly_water_response.raise_for_status()
fortnightly_water_geojson = fortnightly_water_response.json()
fortnightly_water = gpd.GeoDataFrame.from_features(fortnightly_water_geojson["features"], crs="EPSG:4326")
fortnightly_water.drop(columns="geometry").head()


## Read the field descriptions

STAC records describe the published fields. This table selects the fields used below and keeps their original descriptions.


In [ ]:
item_name = f"{state}_{district}_{tehsil}_water_balance_fortnightly_vector"
item_url = urljoin(STAC_URL, f"{state}/{district}/{tehsil}/{item_name}/{item_name}.json")
item_response = requests.get(item_url, timeout=90)
item_response.raise_for_status()
item = item_response.json()
field_notes = pd.DataFrame(item["properties"]["table:columns"])
field_notes.loc[field_notes["name"].isin(['uid', '2017-07-01']), ["name", "type", "description"]]


## Make a table of fortnightly water values

Use the date fields for 2017–18 through 2024–25. Converting the index to dates makes time-series plotting and seasonal grouping straightforward.


In [ ]:
fortnight_row = fortnightly_water.loc[fortnightly_water["uid"] == mws_id].reset_index(drop=True).reindex([0]).iloc[0]
date_fields = sorted(fortnightly_water.filter(regex=r"^\d{4}-\d{2}-\d{2}$").columns)
fortnightly = pd.DataFrame({date: json.loads(fortnight_row[date]) if pd.notna(fortnight_row[date]) else {} for date in date_fields}).T
fortnightly = fortnightly.reindex(columns=["Precipitation", "ET", "RunOff"])
fortnightly = fortnightly[["Precipitation", "ET", "RunOff"]].rename(columns={"Precipitation": "Rainfall", "RunOff": "Runoff"})
fortnightly.index = pd.to_datetime(fortnightly.index)
fortnightly = fortnightly.loc["2017-07-01":"2025-06-30"]
fortnightly.index.name = "Date"
fortnightly.head(12)


## Group the values into seasons

Kharif is July–October, Rabi November–February, and Zaid March–June. These sums use the available recorded intervals, so incomplete seasons are not full-season totals. Each interval belongs to the season of its start date, as in the tehsil API. Annual and fortnightly layers are separate estimates, so their totals can differ.


In [ ]:
seasonal_rows = fortnightly.copy()
month = seasonal_rows.index.month
seasonal_rows["Year"] = seasonal_rows.index.year - (month < 7).astype(int)
seasonal_rows["Season"] = pd.Series(month, index=seasonal_rows.index).map({
    7: "Kharif", 8: "Kharif", 9: "Kharif", 10: "Kharif",
    11: "Rabi", 12: "Rabi", 1: "Rabi", 2: "Rabi", 3: "Zaid", 4: "Zaid", 5: "Zaid", 6: "Zaid"
})
seasonal = seasonal_rows.groupby(["Year", "Season"])[["Rainfall", "ET", "Runoff"]].sum(min_count=1)
seasonal


## Align annual and seasonal charts

Each column follows one water variable. Annual values are above Kharif, Rabi and Zaid. The four rows share a scale within each column.


In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(12, 10), sharex=True, sharey="col")
for column, variable in enumerate(["Rainfall", "ET", "Runoff"]):
    axes[0, column].plot(YEARS, annual[variable], marker="o")
    axes[0, column].set(title=variable, ylabel="Annual (mm)")
    for row, season in enumerate(["Kharif", "Rabi", "Zaid"], start=1):
        values = seasonal.xs(season, level="Season")[variable].reindex(YEARS)
        axes[row, column].plot(YEARS, values, marker="o")
        axes[row, column].set_ylabel(f"{season} (mm)")
    axes[3, column].set_xlabel("July–June year starting")
plt.tight_layout()
plt.show()


## Plot the fortnightly water series

These are water depths for each recorded interval. Separate plots make the different ranges easier to read.


In [ ]:
axes = fortnightly.plot(subplots=True, figsize=(11, 7), sharex=True, legend=False)
for ax, variable in zip(axes, ["Rainfall", "ET", "Runoff"]):
    ax.set_ylabel(f"{variable} (mm)")
plt.tight_layout()
plt.show()


## Read NDVI on crops

NDVI is a unitless measure of vegetation greenness. The date fields contain numeric values directly.


In [ ]:
ndvi_crop_response = requests.get(layer_urls["ndvi_crop"], timeout=90)
ndvi_crop_response.raise_for_status()
ndvi_crop_geojson = ndvi_crop_response.json()
ndvi_crop = gpd.GeoDataFrame.from_features(ndvi_crop_geojson["features"], crs="EPSG:4326")
ndvi_crop.drop(columns="geometry").head()


## Read NDVI on trees

NDVI is a unitless measure of vegetation greenness. The date fields contain numeric values directly.


In [ ]:
ndvi_tree_response = requests.get(layer_urls["ndvi_tree"], timeout=90)
ndvi_tree_response.raise_for_status()
ndvi_tree_geojson = ndvi_tree_response.json()
ndvi_tree = gpd.GeoDataFrame.from_features(ndvi_tree_geojson["features"], crs="EPSG:4326")
ndvi_tree.drop(columns="geometry").head()


## Read NDVI on shrubs

NDVI is a unitless measure of vegetation greenness. The date fields contain numeric values directly.


In [ ]:
ndvi_shrub_response = requests.get(layer_urls["ndvi_shrub"], timeout=90)
ndvi_shrub_response.raise_for_status()
ndvi_shrub_geojson = ndvi_shrub_response.json()
ndvi_shrub = gpd.GeoDataFrame.from_features(ndvi_shrub_geojson["features"], crs="EPSG:4326")
ndvi_shrub.drop(columns="geometry").head()


## Put the three NDVI series together

Read each layer’s own date columns for the same MWS. Pandas aligns the series by date and leaves a gap where a layer has no value.


In [ ]:
crop_ndvi = ndvi_crop.loc[ndvi_crop["uid"] == mws_id].reset_index(drop=True).reindex([0]).iloc[0].filter(regex=r"^\d{4}-\d{2}-\d{2}$")
tree_ndvi = ndvi_tree.loc[ndvi_tree["uid"] == mws_id].reset_index(drop=True).reindex([0]).iloc[0].filter(regex=r"^\d{4}-\d{2}-\d{2}$")
shrub_ndvi = ndvi_shrub.loc[ndvi_shrub["uid"] == mws_id].reset_index(drop=True).reindex([0]).iloc[0].filter(regex=r"^\d{4}-\d{2}-\d{2}$")
ndvi = pd.DataFrame({"Crops": crop_ndvi, "Trees": tree_ndvi, "Shrubs": shrub_ndvi})
ndvi.index = pd.to_datetime(ndvi.index)
ndvi = ndvi.sort_index().loc["2017-07-01":"2025-06-30"]
ndvi.head(12)


## Plot vegetation greenness

The three plots share the NDVI scale and date axis.


In [ ]:
axes = ndvi.plot(subplots=True, figsize=(11, 7), sharex=True, sharey=True, legend=False)
for ax, cover in zip(axes, ["Crops", "Trees", "Shrubs"]):
    ax.set_ylabel(f"{cover} NDVI")
plt.tight_layout()
plt.show()


## Read groundwater extraction data

This layer describes the stage of groundwater extraction and the assessment category.


In [ ]:
soge_response = requests.get(layer_urls["soge"], timeout=90)
soge_response.raise_for_status()
soge_geojson = soge_response.json()
soge = gpd.GeoDataFrame.from_features(soge_geojson["features"], crs="EPSG:4326")
soge.drop(columns="geometry").head()


## Read the field descriptions

STAC records describe the published fields. This table selects the fields used below and keeps their original descriptions.


In [ ]:
item_name = f"{state}_{district}_{tehsil}_stage_of_groundwater_extraction_vector"
item_url = urljoin(STAC_URL, f"{state}/{district}/{tehsil}/{item_name}/{item_name}.json")
item_response = requests.get(item_url, timeout=90)
item_response.raise_for_status()
item = item_response.json()
field_notes = pd.DataFrame(item["properties"]["table:columns"])
field_notes.loc[field_notes["name"].isin(['sgw_dev_pe', 'class', 'agwd_tot', 'ar_gwr_tot']), ["name", "type", "description"]]


## Groundwater assessment for this MWS

Read the published extraction percentage and class.


In [ ]:
soge.loc[soge["uid"] == mws_id, ["sgw_dev_pe", "class"]].rename(columns={"sgw_dev_pe": "Groundwater extraction (%)", "class": "Assessment class"}).T


## Read the aquifer layer

The aquifer class identifies the main aquifer material. It is context for the water series, not an annual measurement.


In [ ]:
aquifer_response = requests.get(layer_urls["aquifer"], timeout=90)
aquifer_response.raise_for_status()
aquifer_geojson = aquifer_response.json()
aquifer = gpd.GeoDataFrame.from_features(aquifer_geojson["features"], crs="EPSG:4326")
aquifer.drop(columns="geometry").head()


## Read the field descriptions

STAC records describe the published fields. This table selects the fields used below and keeps their original descriptions.


In [ ]:
item_name = f"{state}_{district}_{tehsil}_aquifer_vector"
item_url = urljoin(STAC_URL, f"{state}/{district}/{tehsil}/{item_name}/{item_name}.json")
item_response = requests.get(item_url, timeout=90)
item_response.raise_for_status()
item = item_response.json()
field_notes = pd.DataFrame(item["properties"]["table:columns"])
field_notes.loc[field_notes["name"].isin(['Major_Aqui', 'Principal_', 'Age']), ["name", "type", "description"]]


## Aquifer description

Read the classification and the published aquifer description for the same identifier.


In [ ]:
aquifer.loc[aquifer["uid"] == mws_id, ["aquifer_class", "Major_Aqui", "Principal_", "Age"]].T


## Connect to the CoRE Stack API

Read endpoint specifications at [api-doc.core-stack.org](https://api-doc.core-stack.org). The [public API guide](https://docs.core-stack.org/use-precomputed-data/public-apis/) explains how to register, generate an API key and use it. The key goes in the `X-API-Key` header. This cell reads `CORE_STACK_API_KEY` from your environment, or asks for it without showing it. The key is not written into the notebook. After each API request, run the collapsed parsing cell: it keeps the raw response text and reads it with `json.loads`, which accepts `NaN` as a missing numeric value. It also shows HTTP errors and skips dependent API cells if the request fails. Expand the cell to inspect the code.


In [ ]:
from inspect import isawaitable

api_key = os.environ.get("CORE_STACK_API_KEY", "").strip()
if not api_key:
    api_key = getpass("CoRE Stack API key: ")
    if isawaitable(api_key):
        api_key = await api_key

api_headers = {"X-API-Key": str(api_key).strip()}


## Fortnightly water and NDVI from the API

`get_mws_data` returns these time series for one MWS. Read its `time_series` list as a table.


In [ ]:
response = requests.get(API_URL + "get_mws_data/", params={**place, "mws_id": mws_id}, headers=api_headers, timeout=90)
# The next cell checks the HTTP status and reads the response.


In [ ]:
raw_api_data_string = response.text
api_payload = None
if response.ok:
    try:
        api_payload = json.loads(raw_api_data_string.lstrip("\ufeff"))
    except ValueError:
        print("The API returned a response that is not valid JSON. Preview:", raw_api_data_string[:500])
else:
    print(f"API returned HTTP {response.status_code} for {response.url}")
    print("This request did not return data. Other API examples can still be run.")
    print(raw_api_data_string[:500])


In [ ]:
if api_payload is not None:
    api_time_series = pd.DataFrame(api_payload["time_series"])
    display(api_time_series[["date", "precipitation", "runoff", "et", "ndvi_crop", "ndvi_tree", "ndvi_shrub"]].head(12))


## Plot the API time series

Rainfall, runoff and ET share millimetre units. NDVI is shown separately. Change the selected columns to explore another series.


In [ ]:
if api_payload is not None:
    api_time_series["date"] = pd.to_datetime(api_time_series["date"])
    api_series = api_time_series.set_index("date").sort_index()
    api_series[["precipitation", "runoff", "et"]].plot(subplots=True, figsize=(10, 7), ylabel="mm")
    plt.tight_layout()
    plt.show()
    api_series[["ndvi_crop", "ndvi_tree", "ndvi_shrub"]].plot(figsize=(10, 4), ylabel="NDVI")
    display(plt.show())


## Read the tehsil tables from the API

`get_tehsil_data` returns a dictionary of tables for the same place. Here we make the request once and reuse the returned tables below.


In [ ]:
api_response = requests.get(API_URL + "get_tehsil_data/", params=place, headers=api_headers, timeout=180)
# The next cell checks the HTTP status and reads the response.


In [ ]:
raw_api_data_string = api_response.text
api_payload = None
if api_response.ok:
    try:
        api_payload = json.loads(raw_api_data_string.lstrip("\ufeff"))
    except ValueError:
        print("The API returned a response that is not valid JSON. Preview:", raw_api_data_string[:500])
else:
    print(f"API returned HTTP {api_response.status_code} for {api_response.url}")
    print("This request did not return data. Other API examples can still be run.")
    print(raw_api_data_string[:500])


In [ ]:
if api_payload is not None:
    api_data = api_payload
    display(pd.DataFrame({"Table": api_data.keys(), "Rows": [len(rows) for rows in api_data.values()]}))


## Read a water table from the API

The first ten MWS identifiers are shown below. Choose `hydrological_annual`, `hydrological_seasonal`, `soge_vector` or `aquifer_vector`, then select the same MWS used above.


In [ ]:
if api_payload is not None:
    display(pd.DataFrame(api_data['mws'])[['uid']].head(10))
    table_name = 'hydrological_annual'
    api_water_table = pd.DataFrame(api_data[table_name])
    display(api_water_table.loc[api_water_table['uid'] == mws_id].T)
